In [1]:
"""
==============================================================================
MONITOR DA INFLAÇÃO ALIMENTAR POR FAIXA DE RENDA
==============================================================================
"""

import os
import re
import subprocess
import time
import jinja2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# ==============================================================================
# CONFIGURAÇÕES E CONSTANTES
# ==============================================================================
BASE_DIR = r"C:\Users\ITAIGUARA\Dropbox\Relatorio_IPCA_Domicilio_Renda"

ESTRATOS_IPEA = [
    "Renda Muito Baixa", "Renda Baixa", "Renda Média-Baixa",
    "Renda Média", "Renda Média-Alta", "Renda Alta"
]

ESTRATOS_ROTULOS = [
    "Renda Muito\nBaixa", "Renda\nBaixa", "Renda\nMédia-Baixa",
    "Renda\nMédia", "Renda\nMédia-Alta", "Renda\nAlta"
]

CORES_ESTRATOS = [
    "#c0392b", "#d35400", "#8e44ad", "#7f8c8d", "#27ae60", "#1b4f72"
]
COR_IBGE = "#2c3e50"

BLOCOS_ORDEM = [
    "Cereais e Grãos", "Frutas e Hortaliças", "Proteínas Animais",
    "Laticínios", "Óleos e Gorduras", "Industrializados"
]

def fmt_ptbr(valor, casas=2, sinal=False):
    if pd.isna(valor) or valor is None:
        return "-"
    prefixo = "+" if (sinal and valor > 0) else ""
    return f"{prefixo}{valor:.{casas}f}".replace(".", ",")

def acumular_taxas(taxas_pct):
    fatores = [1.0 + (t / 100.0) for t in taxas_pct if not np.isnan(t)]
    if not fatores:
        return 0.0
    return (np.prod(fatores) - 1.0) * 100.0

def calcular_ewma_anualizada(df_mensal, colunas, span=12):
    fatores = 1.0 + (df_mensal[colunas] / 100.0)
    log_fatores = np.log(fatores)
    log_ewma_12 = log_fatores.ewm(span=span, adjust=False).mean() * 12
    return (np.exp(log_ewma_12) - 1.0) * 100.0

def atribuir_bloco(nome):
    cod = re.findall(r'^\d+', str(nome))
    if not cod: return "Industrializados"
    prefixo4 = cod[0][:4]
    if prefixo4 == "1101": return "Cereais e Grãos"
    elif prefixo4 in ["1107", "1108", "1110"]: return "Proteínas Animais"
    elif prefixo4 == "1111": return "Laticínios"
    elif prefixo4 == "1113": return "Óleos e Gorduras"
    elif prefixo4 in ["1103", "1105", "1106"]: return "Frutas e Hortaliças"
    else: return "Industrializados"

def consultar_sidra_com_retry(url, tentativas=3, espera=3):
    for i in range(tentativas):
        try:
            resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
            if resp.status_code == 200:
                dados = resp.json()
                if len(dados) > 1: return dados
        except Exception as e:
            time.sleep(espera * (2 ** i))
    raise ConnectionError("Erro crítico: Falha ao consultar o SIDRA/IBGE.")

# ==============================================================================
# PROCESSAMENTO DE DADOS E CÁLCULOS
# ==============================================================================
def carregar_dados_analiticos():
    arq_mensal = os.path.join(BASE_DIR, "inflacao_mensal_ipca_renda.parquet")
    arq_subitens = os.path.join(BASE_DIR, "ipca_alimentacao_subitens_historico_completo.parquet")
    arq_pesos = os.path.join(BASE_DIR, "pesos_pof_normalizados.parquet")

    df_mensal = pd.read_parquet(arq_mensal)
    df_mensal["data_base"] = pd.to_datetime(df_mensal["data_base"], dayfirst=True, errors="coerce")
    df_mensal = df_mensal.dropna(subset=["data_base"]).sort_values("data_base").reset_index(drop=True)

    df_semestre = df_mensal.tail(6).copy().reset_index(drop=True)
    datas_semestre = df_semestre["data_base"]
    data_ult = datas_semestre.iloc[-1]
    p_ini = datas_semestre.min().strftime("%Y%m")
    p_fim = datas_semestre.max().strftime("%Y%m")

    url_oficial = f"https://apisidra.ibge.gov.br/values/t/7060/n1/all/v/63/p/{p_ini}-{p_fim}/c315/7169,7171/d/v63%202"
    dados_json = consultar_sidra_com_retry(url_oficial)
    df_raw = pd.DataFrame(dados_json[1:])
    col_mes = "D3C" if "D3C" in df_raw.columns else "D2C"
    col_classif = "D4C" if "D4C" in df_raw.columns else "D3C"

    df_of = pd.DataFrame({
        "data_base": pd.to_datetime(df_raw[col_mes], format="%Y%m", errors="coerce"),
        "subitem_id": df_raw[col_classif].astype(str),
        "variacao": pd.to_numeric(df_raw["V"], errors="coerce")
    })
    piv = df_of.pivot_table(index="data_base", columns="subitem_id", values="variacao", aggfunc="first").reindex(datas_semestre)
    serie_ibge_dom = piv["7171"].fillna(0.0).tolist()
    serie_ibge_geral = piv["7169"].fillna(0.0).tolist()

    df_ano = df_mensal[df_mensal["data_base"] >= pd.Timestamp(year=data_ult.year, month=1, day=1)].copy()
    df_12m = df_mensal.tail(12).copy()

    df_ewma_serie = calcular_ewma_anualizada(df_mensal, ESTRATOS_IPEA, span=12)
    df_mensal_com_ewma = df_mensal.copy()
    for col in ESTRATOS_IPEA:
        df_mensal_com_ewma[f"{col}_ewma"] = df_ewma_serie[col]

    p_ini_12m = df_12m["data_base"].min().strftime("%Y%m")
    url_12m_ibge = f"https://apisidra.ibge.gov.br/values/t/7060/n1/all/v/63/p/{p_ini_12m}-{p_fim}/c315/7169,7171/d/v63%202"
    dados12 = consultar_sidra_com_retry(url_12m_ibge)
    df_raw12 = pd.DataFrame(dados12[1:])
    c_mes = "D3C" if "D3C" in df_raw12.columns else "D2C"
    c_item = "D4C" if "D4C" in df_raw12.columns else "D3C"

    ibge_geral_12m_dict, ibge_dom_12m_dict = {}, {}
    for _, row_b in df_raw12.iterrows():
        dt = pd.to_datetime(row_b[c_mes], format="%Y%m", errors="coerce")
        v_num = pd.to_numeric(row_b["V"], errors="coerce")
        if str(row_b[c_item]) == "7169": ibge_geral_12m_dict[dt] = v_num
        elif str(row_b[c_item]) == "7171": ibge_dom_12m_dict[dt] = v_num

    taxas_geral_12m = [ibge_geral_12m_dict.get(d, 0.0) for d in df_12m["data_base"]]
    taxas_dom_12m = [ibge_dom_12m_dict.get(d, 0.0) for d in df_12m["data_base"]]
    taxas_geral_ano = [ibge_geral_12m_dict.get(d, 0.0) for d in df_ano["data_base"]]
    taxas_dom_ano = [ibge_dom_12m_dict.get(d, 0.0) for d in df_ano["data_base"]]

    # Tabela 1: Variações Mensais e Acumuladas
    tab1_ipca = {
        "m2": fmt_ptbr(serie_ibge_geral[-3]), "m1": fmt_ptbr(serie_ibge_geral[-2]),
        "atual": fmt_ptbr(serie_ibge_geral[-1]), "ano": fmt_ptbr(acumular_taxas(taxas_geral_ano)),
        "doze": fmt_ptbr(acumular_taxas(taxas_geral_12m))
    }
    tab1_dom = {
        "m2": fmt_ptbr(serie_ibge_dom[-3]), "m1": fmt_ptbr(serie_ibge_dom[-2]),
        "atual": fmt_ptbr(serie_ibge_dom[-1]), "ano": fmt_ptbr(acumular_taxas(taxas_dom_ano)),
        "doze": fmt_ptbr(acumular_taxas(taxas_dom_12m))
    }

    tabela_variacoes_ipea = []
    for estrato in ESTRATOS_IPEA:
        tabela_variacoes_ipea.append({
            "estrato": estrato, "m2": fmt_ptbr(df_semestre[estrato].iloc[-3]),
            "m1": fmt_ptbr(df_semestre[estrato].iloc[-2]), "atual": fmt_ptbr(df_semestre[estrato].iloc[-1]),
            "ano": fmt_ptbr(acumular_taxas(df_ano[estrato].tolist())),
            "doze": fmt_ptbr(acumular_taxas(df_12m[estrato].tolist()))
        })

    # Tabela 2: Tendências Anualizadas (Mês atual e EWMA)
    tabela2_tendencias = []
    for estrato in ESTRATOS_IPEA:
        val_mes = df_semestre[estrato].iloc[-1]
        val_anualizado_mes = ((1.0 + (val_mes / 100.0)) ** 12 - 1.0) * 100.0
        val_ewma = df_mensal_com_ewma[f"{estrato}_ewma"].iloc[-1]
        tabela2_tendencias.append({
            "estrato": estrato,
            "mes_anualizado": fmt_ptbr(val_anualizado_mes, casas=2, sinal=True),
            "ewma": fmt_ptbr(val_ewma, casas=2, sinal=True)
        })

    # Extração de Subitens
    df_sub_bruto = pd.read_parquet(arq_subitens)
    df_pesos = pd.read_parquet(arq_pesos)
    df_sub_bruto["data_base"] = pd.to_datetime(df_sub_bruto["data_base"], dayfirst=True, errors="coerce")
    df_mes_recente = df_sub_bruto[(df_sub_bruto["data_base"].dt.year == data_ult.year) & (df_sub_bruto["data_base"].dt.month == data_ult.month)].copy()
    
    df_mes_recente["bloco"] = df_mes_recente["subitem_nome"].apply(atribuir_bloco)
    df_mes_recente["codigo_subitem"] = df_mes_recente["subitem_nome"].str.extract(r'^(\d+)')[0].astype(str).str.strip()
    df_pesos["codigo_subitem"] = df_pesos["codigo_subitem"].astype(str).str.strip()
    df_analise = df_mes_recente.merge(df_pesos, on="codigo_subitem", how="left").fillna(0.0)

    matriz_decomp = {}
    for f in ESTRATOS_IPEA:
        fator_p = 1.0 if df_analise[f].sum() <= 1.5 else (1.0 / 100.0)
        df_analise[f"contrib_{f}"] = df_analise["variacao_preco_t"] * (df_analise[f] * fator_p)
        matriz_decomp[f] = {b: round(df_analise.groupby("bloco")[f"contrib_{f}"].sum().get(b, 0.0), 2) for b in BLOCOS_ORDEM}

    tabela_completa_subitens = []
    for f in ESTRATOS_IPEA:
        col_c = f"contrib_{f}"
        top_pos = df_analise.sort_values(col_c, ascending=False).head(5)
        top_neg = df_analise.sort_values(col_c, ascending=True).head(5)
        linhas_faixa = []
        for i in range(min(5, max(len(top_pos), len(top_neg)))):
            r_pos = top_pos.iloc[i] if i < len(top_pos) else None
            r_neg = top_neg.iloc[i] if i < len(top_neg) else None
            linhas_faixa.append({
                "pos": i + 1,
                "item_pos": re.sub(r'^\d+\s*[\.\-]\s*', '', str(r_pos["subitem_nome"])).strip().capitalize() if r_pos is not None else "-",
                "var_pos": fmt_ptbr(r_pos["variacao_preco_t"], sinal=True) if r_pos is not None else "-",
                "contrib_pos": fmt_ptbr(r_pos[col_c], casas=4, sinal=True) if r_pos is not None else "-",
                "item_neg": re.sub(r'^\d+\s*[\.\-]\s*', '', str(r_neg["subitem_nome"])).strip().capitalize() if r_neg is not None else "-",
                "var_neg": fmt_ptbr(r_neg["variacao_preco_t"], sinal=False) if r_neg is not None else "-",
                "contrib_neg": fmt_ptbr(r_neg[col_c], casas=4, sinal=False) if r_neg is not None else "-"
            })
        tabela_completa_subitens.append({"faixa_nome": f, "linhas": linhas_faixa})

    mapa_meses = {1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr", 5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago", 9: "Set", 10: "Out", 11: "Nov", 12: "Dez"}
    mapa_extenso = {1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto", 9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"}
    
    # Identificação do Mês Anterior
    mes_ant_data = datas_semestre.iloc[-2]
    mes_anterior_completo = f"{mapa_extenso[mes_ant_data.month]} de {mes_ant_data.year}"

    return {
        "df_semestre": df_semestre, "rotulos_x": [f"{mapa_meses[d.month]}/{d.year}" for d in datas_semestre],
        "serie_ibge_dom": serie_ibge_dom, "serie_ibge_geral": serie_ibge_geral, "matriz_decomp": matriz_decomp,
        "tabela_completa_subitens": tabela_completa_subitens, "tabela_variacoes_ipea": tabela_variacoes_ipea,
        "tab1_ipca": tab1_ipca, "tab1_dom": tab1_dom,
        "tabela2_tendencias": tabela2_tendencias,
        "col_mes_m2": f"{mapa_meses[datas_semestre.iloc[-3].month]}./{str(datas_semestre.iloc[-3].year)[2:]}",
        "col_mes_m1": f"{mapa_meses[datas_semestre.iloc[-2].month]}./{str(datas_semestre.iloc[-2].year)[2:]}",
        "col_mes_atual": f"{mapa_meses[data_ult.month]}./{str(data_ult.year)[2:]}",
        "mes_ref_nome": f"{mapa_extenso[data_ult.month]} de {data_ult.year}",
        "mes_extenso": mapa_extenso[data_ult.month], "ano_ref": data_ult.year, "data_ult": data_ult,
        "mes_anterior_completo": mes_anterior_completo,
        "tabela_faixas_renda": [
            {"nome": "1 -- Renda muito baixa", "renda_base": r"Menor que R\$ 900,00", "renda_atual": r"Menor que R\$ 2.299,82"},
            {"nome": "2 -- Renda baixa", "renda_base": r"De R\$ 900,00 a R\$ 1.350,00", "renda_atual": r"De R\$ 2.299,82 a R\$ 3.449,73"},
            {"nome": "3 -- Renda média-baixa", "renda_base": r"De R\$ 1.350,01 a R\$ 2.250,00", "renda_atual": r"De R\$ 3.449,74 a R\$ 5.749,55"},
            {"nome": "4 -- Renda média", "renda_base": r"De R\$ 2.250,01 a R\$ 4.500,00", "renda_atual": r"De R\$ 5.749,56 a R\$ 11.499,11"},
            {"nome": "5 -- Renda média-alta", "renda_base": r"De R\$ 4.500,01 a R\$ 9.000,00", "renda_atual": r"De R\$ 11.499,12 a R\$ 22.998,22"},
            {"nome": "6 -- Renda alta", "renda_base": r"Maior que R\$ 9.000,00", "renda_atual": r"Maior que R\$ 22.998,22"}
        ], "faixas_renda_base_ano": "jan./2009", "faixas_renda_atual_ano": f"jan./{data_ult.year}"
    }

# ==============================================================================
# GERAÇÃO DA NARRATIVA EDITORIAL 
# ==============================================================================
def gerar_narrativa_dinamica(dados):
    df_sem = dados["df_semestre"]
    baixo_val = df_sem["Renda Muito Baixa"].iloc[-1]
    alto_val = df_sem["Renda Alta"].iloc[-1]
    media_val = df_sem["Renda Média"].iloc[-1]
    dom_val = dados["serie_ibge_dom"][-1]

    quote_economista = (
        f"Em setembro, a inflação da alimentação no domicílio foi positiva em todos os estratos de renda, "
        f"mas apresentou diferenças de intensidade. A variação foi de {fmt_ptbr(baixo_val)}\\% na Renda Muito Baixa "
        f"e de {fmt_ptbr(alto_val)}\\% na Renda Alta, enquanto a Renda Média registrou a maior taxa mensal, "
        f"de {fmt_ptbr(media_val)}\\%, acima do IPCA Geral e do IPCA de Alimentação no Domicílio."
    )
    pontos_chave = [
        f"Diferenças de intensidade na inflação alimentar em {dados['mes_ref_nome']}: a Renda Muito Baixa registrou variação de {fmt_ptbr(baixo_val, sinal=True)}\\%, enquanto a Renda Alta anotou {fmt_ptbr(alto_val, sinal=True)}\\% (diferença de {fmt_ptbr(abs(alto_val - baixo_val))} p.p. entre os extremos).",
        f"O IPCA de Alimentação no Domicílio, apurado pelo IBGE, registrou alta de {fmt_ptbr(dom_val, sinal=True)}\\%, igual à estimada para o estrato de Renda Alta.",
        "A contribuição negativa de Proteínas Animais atenuou a inflação, sobretudo nos estratos de Renda Média-Alta (-0,11 p.p.) e Renda Alta (-0,07 p.p.).",
        "Frutas e Hortaliças foi o principal vetor de alta em todos os estratos, com contribuição de +0,67 p.p. na Renda Muito Baixa e de +1,09 p.p. na Renda Média."
    ]
    texto_p1 = f"Os dados de {dados['mes_ref_nome']} mostram a retomada das altas nos preços da alimentação no domicílio na maioria dos estratos e a manutenção de variações positivas em todas as faixas de renda. Conforme ilustra o painel à direita, a Renda Média registrou a maior inflação mensal ({fmt_ptbr(media_val)}\\%), seguida pela Renda Alta ({fmt_ptbr(alto_val)}\\%) e pela Renda Muito Baixa ({fmt_ptbr(baixo_val)}\\%). A trajetória semestral, apresentada no gráfico à esquerda, evidencia a recuperação das taxas após as quedas observadas em julho."
    texto_p2 = "A decomposição das contribuições mensais mostra que Frutas e Hortaliças exerceu pressão positiva sobre a inflação em todos os estratos de renda, com maior contribuição na Renda Média (+1,09 p.p.), seguida pela Renda Alta (+0,89 p.p.) e pela Renda Média-Alta (+0,87 p.p.). Em sentido contrário, Proteínas Animais contribuiu negativamente para a variação mensal, atenuando as taxas na Renda Média-Alta (-0,11 p.p.) e na Renda Alta (-0,07 p.p.)."
    
    return quote_economista, pontos_chave, texto_p1, texto_p2

# ==============================================================================
# GERAÇÃO DOS GRÁFICOS
# ==============================================================================
def gerar_figura1_trajetoria_barras(dados, output_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.4), dpi=300)
    plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
    df_sem, n_meses = dados["df_semestre"], len(dados["rotulos_x"])
    
    ax1 = axes[0]
    marcadores, estilos = ['o', 's', '^', 'v', 'D', 'o'], ['-', '-', '--', '--', '-.', '-']
    pontos_finais = []
    for i, estrato in enumerate(ESTRATOS_IPEA):
        valores = df_sem[estrato].tolist()
        ax1.plot(range(n_meses), valores, color=CORES_ESTRATOS[i], linestyle=estilos[i], marker=marcadores[i], markersize=4.6, linewidth=1.7, label=estrato)
        pontos_finais.append({'valor': valores[-1], 'cor': CORES_ESTRATOS[i]})

    val_ibge_dom = dados["serie_ibge_dom"]
    ax1.plot(range(n_meses), val_ibge_dom, color=COR_IBGE, linestyle=':', marker='X', markersize=5.5, linewidth=1.9, label="IPCA - Alim. Domicílio")
    pontos_finais.append({'valor': val_ibge_dom[-1], 'cor': COR_IBGE})
    
    pontos_ord = sorted(pontos_finais, key=lambda x: x['valor'])
    pos_y = [p['valor'] for p in pontos_ord]
    for _ in range(100):
        for i in range(len(pos_y) - 1):
            if pos_y[i+1] - pos_y[i] < 0.13:
                ajuste = (0.13 - (pos_y[i+1] - pos_y[i])) * 0.5
                pos_y[i] -= ajuste
                pos_y[i+1] += ajuste

    for item, y_adj in zip(pontos_ord, pos_y):
        ax1.annotate(fmt_ptbr(item['valor'], sinal=True), xy=(n_meses - 1 + 0.08, y_adj), xycoords='data', va='center', ha='left', fontsize=7.3, fontweight='bold', color=item['cor'])

    ax1.set_title("Trajetória Semestral: Estratos e IPCA Alimentação no Domicílio (%)", fontsize=9.0, fontweight='bold', pad=10)
    ax1.set_xticks(range(n_meses))
    ax1.set_xticklabels(dados["rotulos_x"], fontsize=8.0)
    ax1.set_xlim(-0.15, n_meses - 1 + 0.65)
    ax1.axhline(0, color='#95a5a6', linestyle='-', linewidth=0.7, alpha=0.6)
    ax1.grid(linestyle="--", alpha=0.4, color="#B0B5BA")
    ax1.legend(frameon=False, fontsize=6.8, loc='lower left', ncol=2)

    ax2 = axes[1]
    valores_barras = [df_sem[e].iloc[-1] for e in ESTRATOS_IPEA] + [val_ibge_dom[-1]]
    barras = ax2.bar(ESTRATOS_ROTULOS + ["IPCA\nDomicílio"], valores_barras, color=CORES_ESTRATOS + [COR_IBGE], width=0.56, alpha=0.92)
    ax2.set_title("Inflação Alimentar Mensal por Faixa de Renda (%)", fontsize=9.0, fontweight="bold", pad=10)
    margem = max(abs(max(valores_barras)), abs(min(valores_barras))) * 0.35
    ax2.set_ylim(min(min(valores_barras) - margem, -0.2), max(max(valores_barras) + margem, 0.2))
    ax2.axhline(0, color="#14233C", linewidth=0.8)
    ax2.grid(axis="y", linestyle="--", alpha=0.4, color="#B0B5BA")
    ax2.tick_params(axis="x", labelsize=7.0)
    
    for b in barras:
        h = b.get_height()
        ax2.annotate(fmt_ptbr(h, sinal=True), xy=(b.get_x() + b.get_width() / 2, h + (0.03 if h >= 0 else -0.07)), ha="center", va="bottom" if h >= 0 else "top", fontsize=7.2, fontweight="bold")
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "painel_1x2_trajetoria_barras.pdf"), format="pdf")
    plt.close()

def gerar_figura2_contribuicao_6faixas(dados, output_dir):
    fig, axes = plt.subplots(2, 3, figsize=(14.2, 7.6), dpi=300)
    plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
    ordem_invertida = list(reversed(BLOCOS_ORDEM))
    limite_x = max(max([abs(v) for f in ESTRATOS_IPEA for v in dados["matriz_decomp"][f].values()]), 0.15) * 1.30

    for idx, estrato in enumerate(ESTRATOS_IPEA):
        ax = axes[idx // 3, idx % 3]
        valores = [dados["matriz_decomp"][estrato][b] for b in ordem_invertida]
        barras = ax.barh(ordem_invertida, valores, color=CORES_ESTRATOS[idx], height=0.68, alpha=0.90, edgecolor=CORES_ESTRATOS[idx], linewidth=0.6)
        ax.axvline(0, color="#14233C", linewidth=0.8, alpha=0.7)
        ax.set_title(estrato, fontsize=10.2, fontweight="bold", pad=8, color="#14233C")
        ax.set_xlabel("Contribuição (p.p.)", fontsize=7.8, fontweight="bold")
        ax.grid(axis="x", linestyle="--", alpha=0.35, color="#B0B5BA")
        ax.tick_params(labelsize=7.8)
        ax.set_xlim(-limite_x, limite_x)

        for bar, val in zip(barras, valores):
            ax.text(val + (limite_x * 0.04 if val >= 0 else -limite_x * 0.04), bar.get_y() + bar.get_height() / 2, fmt_ptbr(val, sinal=True), va="center", ha="left" if val >= 0 else "right", fontsize=7.5, fontweight="bold", color="#14233C")
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "painel_contribuicao_6faixas.pdf"), format="pdf")
    plt.close()

def compilar_relatorio(output_dir, tex_nome):
    cmd = ["pdflatex", "-interaction=nonstopmode", tex_nome]
    subprocess.run(cmd, cwd=output_dir, capture_output=True, text=True)
    res = subprocess.run(cmd, cwd=output_dir, capture_output=True, text=True)
    if res.returncode == 0: print(f"\n[SUCESSO] PDF compilado em: {output_dir}")
    else: print("\n[FALHA NA COMPILAÇÃO LATEX]")

# ==============================================================================
# EXECUÇÃO DO FLUXO COMPLETO E EXPORTAÇÃO
# ==============================================================================
template_path = os.path.join(BASE_DIR, "template.tex")
if not os.path.exists(template_path):
    raise FileNotFoundError(f"Arquivo '{template_path}' não encontrado. Salve o código LaTeX com este nome exato.")

dados = carregar_dados_analiticos()

sufixo_mes_ano = f"{dados['mes_extenso'].lower()}_{dados['ano_ref']}"
nome_pasta = f"edicao_{sufixo_mes_ano}"
output_dir = os.path.join(BASE_DIR, nome_pasta)
os.makedirs(output_dir, exist_ok=True)

gerar_figura1_trajetoria_barras(dados, output_dir)
gerar_figura2_contribuicao_6faixas(dados, output_dir)

df_sem = dados["df_semestre"]
baixo_val = df_sem["Renda Muito Baixa"].iloc[-1]
alto_val = df_sem["Renda Alta"].iloc[-1]
dom_val = dados["serie_ibge_dom"][-1]
geral_val = dados["serie_ibge_geral"][-1]

quote_economista, pontos_chave, texto_analitico_p1, texto_analitico_p2 = gerar_narrativa_dinamica(dados)

tag_mes_ano = dados["data_ult"].strftime("%m_%Y")
edicao_texto = f"Por Estrato de Renda -- {dados['mes_ref_nome']}"
chamada_capa = "A dinâmica dos preços alimentares nos diferentes estratos de renda familiar"
data_divulgacao_texto = "Divulgado em 09 de outubro de 2026."

pasta_dados = os.path.join(BASE_DIR, "dados")
os.makedirs(pasta_dados, exist_ok=True)

df_var_completo = pd.read_parquet(os.path.join(BASE_DIR, "inflacao_mensal_ipca_renda.parquet"))
df_ind_completo = pd.DataFrame(index=df_var_completo.index, columns=df_var_completo.columns)
df_ind_completo["data_base"] = df_var_completo["data_base"]
for c in [col for col in df_var_completo.columns if col != "data_base"]:
    df_ind_completo.loc[0, c] = 100.00
    df_ind_completo.loc[1:, c] = (100.00 * (1.0 + (df_var_completo[c].iloc[1:] / 100.0)).cumprod()).round(2)

df_var_completo.to_excel(os.path.join(pasta_dados, f"inflacao_mensal_ipca_renda_{tag_mes_ano}.xlsx"), sheet_name="Variacao_Mensal", index=False)
df_var_completo.to_csv(os.path.join(pasta_dados, f"inflacao_mensal_ipca_renda_{tag_mes_ano}.csv"), index=False, sep=";", decimal=",")
df_var_completo.to_parquet(os.path.join(pasta_dados, f"inflacao_mensal_ipca_renda_{tag_mes_ano}.parquet"), index=False)
df_ind_completo.to_excel(os.path.join(pasta_dados, f"indices_base100_ipca_renda_{tag_mes_ano}.xlsx"), sheet_name="Base100", index=False)
df_ind_completo.to_csv(os.path.join(pasta_dados, f"indices_base100_ipca_renda_{tag_mes_ano}.csv"), index=False, sep=";", decimal=",")
df_ind_completo.to_parquet(os.path.join(pasta_dados, f"indices_base100_ipca_renda_{tag_mes_ano}.parquet"), index=False)

link_github_raw = "https://raw.githubusercontent.com/Itaiguara-Bezerra/itaiguara-bezerra.github.io/main/dados"
link_edicao_anterior = "https://raw.githubusercontent.com/Itaiguara-Bezerra/itaiguara-bezerra.github.io/main/edicoes_anteriores/Monitor_Inflacao_Alimentar_anterior.pdf"

# A análise da tendência para alocar após a Figura 1 na Página 2
texto_analise_ewma = "A suavização exponencial (EWMA) atua como um filtro analítico para extrair a tendência subjacente da inflação alimentar, atenuando a forte volatilidade climática e sazonal da ponta. Conforme detalhado na Tabela 2 (página seguinte), o método evidencia a persistência inflacionária nos estratos, isolando os choques transitórios."

dados_contexto = {
    "edicao_texto": edicao_texto, "chamada_capa": chamada_capa, "mes_ano_referencia": dados["mes_ref_nome"],
    "autor_nome": "Itaiguara de Oliveira Bezerra", "autor_titulo": "Economista Sênior e de Dados",
    "autor_afiliacao": "Pesquisador Independente", "autor_contato": "itaiguara@terra.com.br",
    "autor_website": "itaiguara-bezerra.github.io", "autor_divulgacao": data_divulgacao_texto,
    "ipca_geral_mes": fmt_ptbr(dados["serie_ibge_geral"][-1], sinal=True), 
    "ipca_domicilio_mes": fmt_ptbr(dados["serie_ibge_dom"][-1], sinal=True),
    "estrato_baixo_mes": fmt_ptbr(dados["df_semestre"]["Renda Muito Baixa"].iloc[-1], sinal=True), 
    "estrato_alto_mes": fmt_ptbr(dados["df_semestre"]["Renda Alta"].iloc[-1], sinal=True),
    "cor_ipca_geral": "bisterra" if dados["serie_ibge_geral"][-1] > 0 else "bisnavy",
    "cor_ipca_dom": "bisterra" if dados["serie_ibge_dom"][-1] > 0 else "bisnavy",
    "cor_estrato_baixo": "bisterra" if dados["df_semestre"]["Renda Muito Baixa"].iloc[-1] > 0 else "bisnavy",
    "cor_estrato_alto": "bisterra" if dados["df_semestre"]["Renda Alta"].iloc[-1] > 0 else "bisnavy",
    "quote_economista": quote_economista, "pontos_chave": pontos_chave,
    "texto_analitico_p1": texto_analitico_p1, "texto_analitico_p2": texto_analitico_p2,
    "texto_analise_ewma": texto_analise_ewma,
    "col_mes_m2": dados["col_mes_m2"], "col_mes_m1": dados["col_mes_m1"], "col_mes_atual": dados["col_mes_atual"],
    "tab1_ipca": dados["tab1_ipca"], "tab1_dom": dados["tab1_dom"],
    "tabela_variacoes_ipea": dados["tabela_variacoes_ipea"], "tabela2_tendencias": dados["tabela2_tendencias"],
    "tabela_completa_subitens": dados["tabela_completa_subitens"], "tabela_faixas_renda": dados["tabela_faixas_renda"],
    "faixas_renda_base_ano": dados["faixas_renda_base_ano"], "faixas_renda_atual_ano": dados["faixas_renda_atual_ano"],
    "mes_anterior_completo": dados["mes_anterior_completo"],
    "link_edicao_anterior": link_edicao_anterior,
    "link_drive_nota": f"{link_github_raw}/Nota_Tecnica_IPCA_Alimentacao_No_Domicilio_Por_Estrato_Socioeconomico.pdf",
    "link_drive_xlsx_var": f"{link_github_raw}/inflacao_mensal_ipca_renda_{tag_mes_ano}.xlsx",
    "link_drive_csv_var": f"{link_github_raw}/inflacao_mensal_ipca_renda_{tag_mes_ano}.csv",
    "link_drive_parquet_var": f"{link_github_raw}/inflacao_mensal_ipca_renda_{tag_mes_ano}.parquet",
    "link_drive_xlsx_ind": f"{link_github_raw}/indices_base100_ipca_renda_{tag_mes_ano}.xlsx",
    "link_drive_csv_ind": f"{link_github_raw}/indices_base100_ipca_renda_{tag_mes_ano}.csv",
    "link_drive_parquet_ind": f"{link_github_raw}/indices_base100_ipca_renda_{tag_mes_ano}.parquet",
}

env = jinja2.Environment(block_start_string=r"\BLOCK{", block_end_string=r"}", variable_start_string=r"\VAR{", variable_end_string=r"}", loader=jinja2.FileSystemLoader(BASE_DIR), autoescape=False)
template = env.get_template("template.tex")
tex_renderizado = template.render(dados_contexto)

nome_base_doc = f"Monitor_Inflacao_Alimentar_{tag_mes_ano}"
nome_arquivo_tex = f"{nome_base_doc}.tex"
tex_saida = os.path.join(output_dir, nome_arquivo_tex)

with open(tex_saida, "w", encoding="utf-8") as f:
    f.write(tex_renderizado)

print(f"[OK] Arquivo TeX gerado: {tex_saida}")
compilar_relatorio(output_dir, nome_arquivo_tex)

[OK] Arquivo TeX gerado: C:\Users\ITAIGUARA\Dropbox\Relatorio_IPCA_Domicilio_Renda\edicao_setembro_2026\Monitor_Inflacao_Alimentar_09_2026_v2.tex

[SUCESSO] PDF compilado em: C:\Users\ITAIGUARA\Dropbox\Relatorio_IPCA_Domicilio_Renda\edicao_setembro_2026
